### Building Your First "AI Memory" with ChromaDB
In this notebook, we will build a RAG (Retrieval-Augmented Generation) system.

The Problem: LLMs (like Gemini or GPT) have a cutoff date and don't know your private data.

The Solution: RAG. We give the AI a "library" (Vector Database) to look up facts before it speaks.

In [3]:
# Install the library (run this in a cell)
!pip install chromadb -q

import chromadb
print("ChromaDB is ready!")


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ChromaDB is ready!


### Understanding Embeddings
Before we code, you must understand the Embedding. Computers don't understand words; they understand math. An embedding turns a sentence into a long list of numbers (a vector).

The Rule of Similarity: If two sentences have similar meanings (e.g., "I love dogs" and "Puppies are great"), their numbers will be mathematically "close" to each other in a 3D space.

### Setup the Database
We will create a "Collection." Think of this as a folder in your database where we will store our facts.

In [4]:
# 1. Create a client (this runs in your computer's RAM for now)
client = chromadb.Client()

# 2. Create a collection called "student_handbook"
# This is where our 'knowledge' will live
collection = client.get_or_create_collection(name="university_info")

### Adding Knowledge (The "Library")
Let’s give our AI some specific facts that it wouldn't know otherwise.

In [5]:
# We provide the text, and Chroma handles the math (embeddings) automatically
collection.add(
    documents=[
        "The campus library is open from 8 AM to 10 PM daily.",
        "The pizza place on 5th street has a student discount on Tuesdays.",
        "Professor Smith's office is located in Room 402 of the Science Building.",
        "The final exam for Intro to Physics is on December 12th.",
    ],
    ids=["info_1", "info_2", "info_3", "info_4"],
)

print("Knowledge successfully stored!")

Knowledge successfully stored!


### The Retrieval (The "Search")
Now, we act as the user. We ask a question. Chroma will turn our question into a vector, compare it to our stored facts, and pull the best match.

In [6]:
# Ask a question
query = "When can I get cheap pizza?"

# Retrieve the top 1 result
results = collection.query(query_texts=[query], n_results=1)

# Extract the text from the results
retrieved_text = results["documents"][0][0]

print(f"User Question: {query}")
print(f"Retrieved Fact: {retrieved_text}")

User Question: When can I get cheap pizza?
Retrieved Fact: The pizza place on 5th street has a student discount on Tuesdays.


### The "Generation" (Connecting to an LLM)
In a real RAG app, you don't just show the raw database result. You send that result to an LLM to make it sound natural.

In [7]:
# PREVIEW OF THE FINAL STEP:
prompt = f"""
You are a helpful university assistant. 
Use the following piece of context to answer the user's question.
If the answer isn't in the context, say you don't know.

Context: {retrieved_text}
User Question: {query}

Answer:"""

# You would then send this 'prompt' to an API like Gemini or OpenAI.
# The AI would respond: "You can get a student discount on pizza every Tuesday on 5th street!"

### Conclusion
Embeddings: Turn text into map coordinates.

Vector DB: Stores those coordinates.

Retrieval: Finds the closest coordinate to your question.

Augmentation: Hands that info to the LLM so it doesn't "hallucinate" or guess.

### Integrating the "Full RAG" Logic
Since we have the environment ready, let’s look at the final piece of the RAG puzzle: Connecting the Database to an LLM. In a real classroom setting, you’d use a "Generator" like Gemini. The diagram below shows how the retrieved "context" is sandwiched into a prompt to prevent the AI from making things up.

In [13]:
# 1. The User asks about the OFFICE
query = "Where is Professor Smith's office?"

# 2. Chroma finds the most SIMILAR text
results = collection.query(
    query_texts=[query], n_results=1  # We only want the single best match
)

# 3. Pull the specific document
context_from_db = results["documents"][0][0]
print(f"Retrieved Context: {context_from_db}")

# 4. Now the prompt will make sense!
prompt = f"Context: {context_from_db}\nQuestion: {query}"
print(prompt)

Retrieved Context: Professor Smith's office is located in Room 402 of the Science Building.
Context: Professor Smith's office is located in Room 402 of the Science Building.
Question: Where is Professor Smith's office?
